# Experiment: Baseline Panel Downstream Experiment

这个 notebook 对应 `scripts/trishift/analysis/baseline_panel.py`。

目的：
- 在统一口径下比较 `TriShift nearest/random`、`Scouter`、`GEARS`、`GenePert`、`Systema` baselines。
- 生成可直接用于论文主表和补充表的 summary / ranking / subgroup 结果。

建议：
- `adamson` 先用于 smoke test。
- `norman` 再查看 `subgroup` 分层表现。


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

repo_root = Path.cwd()
if not (repo_root / "scripts").exists() and (repo_root.parent / "scripts").exists():
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))

import scripts.trishift.analysis.baseline_panel as baseline_panel
importlib.reload(baseline_panel)

run_baseline_panel = baseline_panel.run_baseline_panel
DEFAULT_MODEL_REQUESTS = baseline_panel.DEFAULT_MODEL_REQUESTS
PANEL_METRICS = baseline_panel.PANEL_METRICS

repo_root


## Parameters

直接在这里改数据集、模型集合和 split。`systema_root` 一般留空，让脚本自动找最新结果。


In [ ]:
dataset = "adamson"  # adamson | norman | replogle_k562 | replogle_rpe1
models = [
    "trishift_nearest",
    "scouter",
    "gears",
    "genepert",
    "systema_nonctl_mean",
    "systema_matching_mean",
]
split_ids = [1, 2, 3, 4, 5]
out_root = None
systema_root = None

result = run_baseline_panel(
    dataset=dataset,
    models=models,
    split_ids=split_ids,
    out_root=out_root,
    systema_root=systema_root,
)

print(f"out_dir: {result['out_dir']}")
print(f"panel metrics: {PANEL_METRICS}")


## Summary Tables

先看总表和 ranking。`mean_pearson` 仍然是最直接的主排序参考。


In [ ]:
display(result["summary_df"])
display(result["ranking_df"])
if not result["subgroup_df"].empty:
    display(result["subgroup_df"])


## Artifacts

这里直接回看导出的图和文件路径。适合做论文表格/图的二次整理。


In [ ]:
out_dir = Path(result["out_dir"])
display(Image(filename=str(out_dir / "baseline_panel_heatmap.png")))

for path in [
    out_dir / "baseline_panel_raw.csv",
    out_dir / "baseline_panel_summary.csv",
    out_dir / "baseline_panel_ranking.csv",
    out_dir / "norman_subgroup_panel.csv",
    out_dir / "run_meta.json",
]:
    print(path)
